# 02l — YOLOv8 Classification Training with Synthetic Data

**Project:** UREP 32-0210-250078 | Crack Classification

**Architecture:** YOLOv8s-cls (Ultralytics framework, PyTorch-native)

**Experiment:** Same architecture, hyperparameters, and seeds as `02e_training_yolo.ipynb`.
The only difference is that 806 synthetic AutoCAD-generated crack images are
added to the training set (102 debonding/corrosion, 302 flexural, 402 shear).
Val/test sets are identical to the baseline for fair comparison.

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import numpy as np
from ultralytics import YOLO

import config
from src.dataset import prepare_synthetic_split
from src.evaluation import evaluate_predictions
from src.device import set_seed

# Reproducibility
set_seed(config.RANDOM_SEED)

OUTPUT_DIR = os.path.join(config.OUTPUT_DIR, "yolo_synthetic")
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)

# Materialize synthetic split
SPLIT = config.SPLIT_SYNTHETIC_DIR
prepare_synthetic_split()

print(f"Model: YOLOv8s-cls (with Synthetic Data)")
print(f"Data: {SPLIT}")

In [ ]:
# Verify data structure
for subset in ["train", "val", "test"]:
    subset_dir = os.path.join(SPLIT, subset)
    total = sum(len([f for f in os.listdir(os.path.join(subset_dir, c))
                     if os.path.splitext(f)[1].lower() in {".jpg", ".jpeg", ".png", ".bmp"}])
                for c in config.CLASS_NAMES if os.path.isdir(os.path.join(subset_dir, c)))
    print(f"  {subset}: {total:,} images")

In [ ]:
model = YOLO(config.YOLO_MODEL)

results = model.train(
    data=SPLIT,
    epochs=config.YOLO_EPOCHS,
    imgsz=config.YOLO_IMG_SIZE,
    batch=16,
    patience=config.YOLO_PATIENCE,
    lr0=config.YOLO_LR0,
    lrf=config.YOLO_LRF,
    dropout=config.YOLO_DROPOUT,
    optimizer="AdamW",
    degrees=15.0, fliplr=0.5, flipud=0.0, shear=0.0,
    seed=config.RANDOM_SEED,
    project=OUTPUT_DIR, name="train", exist_ok=True, verbose=True,
)

In [ ]:
# Test evaluation — from ORIGINAL split (no synthetic) for fair comparison
best_model_path = os.path.join(OUTPUT_DIR, "train", "weights", "best.pt")
model = YOLO(best_model_path)

test_dir = os.path.join(config.SPLIT_DIR, "test")
y_true, y_pred = [], []

for cls_idx, cls_name in enumerate(config.CLASS_NAMES):
    cls_dir = os.path.join(test_dir, cls_name)
    if not os.path.isdir(cls_dir): continue
    files = [f for f in os.listdir(cls_dir) if os.path.splitext(f)[1].lower() in {".jpg", ".jpeg", ".png", ".bmp"}]
    for fname in files:
        result = model.predict(os.path.join(cls_dir, fname), verbose=False)
        pred_name = result[0].names[result[0].probs.top1]
        pred_cls_idx = config.CLASS_NAMES.index(pred_name) if pred_name in config.CLASS_NAMES else result[0].probs.top1
        y_true.append(cls_idx)
        y_pred.append(pred_cls_idx)
    print(f"  {cls_name}: {len(files)} done")

metrics = evaluate_predictions(np.array(y_true), np.array(y_pred), output_dir=OUTPUT_DIR, model_name="yolo_synth")